# STCALNet Hyperparameter Search Pipeline (Colab)

This notebook runs a two-phase hyperparameter search for `st_calnet_caiso_tuning.py` on Google Colab GPU.

Workflow:
- Phase 1: broad/coarse search
- Phase 2: local refinement around top runs
- Aggregate all completed runs into a leaderboard

Before running:
1. Switch runtime to GPU (`Runtime -> Change runtime type`).
2. Ensure project files are available in `PROJECT_DIR`.
3. Ensure dataset exists at `CSV_PATH`.


In [ ]:
import os
import sys
import json
import time
import random
import itertools
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import torch

# ---- User config ----
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/caiso-forecasting')
LOCAL_PROJECT_DIR = Path('/content/caiso-forecasting')
PROJECT_DIR = DRIVE_PROJECT_DIR if USE_GOOGLE_DRIVE else LOCAL_PROJECT_DIR

CSV_PATH = 'caiso_dataset_20200101_to_20241231.csv'
REGION = 'caiso'
TRAINER_SCRIPT = 'st_calnet_caiso_tuning.py'
OUT_DIR = Path('runs/st_calnet_colab')

PHASE1_LIMIT = 24
PHASE1_MAX_EPOCHS = 40
PHASE1_PATIENCE = 6

PHASE2_LIMIT = 16
PHASE2_MAX_EPOCHS = 80
PHASE2_PATIENCE = 10

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Python:', sys.version)
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'PROJECT_DIR not found: {PROJECT_DIR}. '
        'Check USE_GOOGLE_DRIVE and folder location in MyDrive.'
    )

os.chdir(PROJECT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Working directory:', Path.cwd())
print('Output directory:', OUT_DIR)

# Optional package install cell (uncomment if needed)
# !pip -q install -U numpy pandas scikit-learn torch


In [ ]:
def run_name_from_cfg(prefix: str, idx: int, cfg: dict) -> str:
    drop = str(cfg['dropout']).replace('.', 'p')
    return (
        f"{prefix}_{idx:03d}_seq{cfg['seq_len']}_hid{cfg['lstm_hidden']}"
        f"_lay{cfg['lstm_layers']}_drop{drop}_lr{cfg['learning_rate']}_wd{cfg['weight_decay']}"
    )


def run_experiment(run_name: str, cfg: dict, max_epochs: int, patience: int) -> dict:
    run_dir = OUT_DIR / run_name
    metrics_path = run_dir / 'metrics.json'
    log_path = OUT_DIR / 'tuning_logs' / f'{run_name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)

    if metrics_path.exists():
        return {'run_name': run_name, 'status': 'skipped_existing', 'return_code': 0, 'log_path': str(log_path)}

    cmd = [
        sys.executable, TRAINER_SCRIPT,
        '--csv-path', CSV_PATH,
        '--region', REGION,
        '--out-dir', str(OUT_DIR),
        '--run-name', run_name,
        '--seq-len', str(cfg['seq_len']),
        '--lstm-hidden', str(cfg['lstm_hidden']),
        '--lstm-layers', str(cfg['lstm_layers']),
        '--dropout', str(cfg['dropout']),
        '--learning-rate', str(cfg['learning_rate']),
        '--weight-decay', str(cfg['weight_decay']),
        '--batch-size', str(cfg['batch_size']),
        '--conv-channels', str(cfg['conv_channels']),
        '--attn-dim', str(cfg['attn_dim']),
        '--max-epochs', str(max_epochs),
        '--patience', str(patience),
        '--seed', str(cfg['seed']),
        '--device', DEVICE,
    ]

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, check=False)

    return {
        'run_name': run_name,
        'status': 'done' if proc.returncode == 0 else 'failed',
        'return_code': proc.returncode,
        'log_path': str(log_path),
    }


def collect_rows(out_dir: Path) -> list:
    rows = []
    for mp in sorted(out_dir.glob('*/metrics.json')):
        payload = json.loads(mp.read_text(encoding='utf-8'))
        cfg = payload.get('config', {})
        val = payload.get('metrics', {}).get('validation', {})
        tst = payload.get('metrics', {}).get('test', {})
        rows.append({
            'run_name': payload.get('run_name', mp.parent.name),
            'val_RMSE': val.get('RMSE'),
            'val_MAE': val.get('MAE'),
            'val_MAPE': val.get('MAPE'),
            'test_RMSE': tst.get('RMSE'),
            'test_MAE': tst.get('MAE'),
            'test_MAPE': tst.get('MAPE'),
            'seq_len': cfg.get('seq_len'),
            'lstm_hidden': cfg.get('lstm_hidden'),
            'lstm_layers': cfg.get('lstm_layers'),
            'dropout': cfg.get('dropout'),
            'learning_rate': cfg.get('learning_rate'),
            'weight_decay': cfg.get('weight_decay'),
            'batch_size': cfg.get('batch_size'),
            'seed': cfg.get('seed'),
        })
    return rows


def leaderboard_df(out_dir: Path) -> pd.DataFrame:
    rows = collect_rows(out_dir)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values('val_RMSE').reset_index(drop=True)
    df.insert(0, 'rank', np.arange(1, len(df) + 1))
    return df


In [ ]:
# Phase 1: broad search
random.seed(SEED)

space = list(itertools.product(
    [24, 48, 72, 168],        # seq_len
    [32, 64, 128, 256],       # hidden
    [1, 2, 3],                # layers
    [0.0, 0.1, 0.2, 0.3],     # dropout
    [1e-4, 3e-4, 1e-3, 3e-3], # lr
    [1e-6, 1e-5, 1e-4],       # weight_decay
    [16, 32, 64],             # batch_size
))

random.shuffle(space)
phase1 = space[:PHASE1_LIMIT]

phase1_results = []
start = time.time()
for i, (seq_len, hidden, layers, dropout, lr, wd, bs) in enumerate(phase1):
    cfg = {
        'seq_len': seq_len,
        'lstm_hidden': hidden,
        'lstm_layers': layers,
        'dropout': dropout,
        'learning_rate': lr,
        'weight_decay': wd,
        'batch_size': bs,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + i,
    }
    rn = run_name_from_cfg('p1', i, cfg)
    res = run_experiment(rn, cfg, max_epochs=PHASE1_MAX_EPOCHS, patience=PHASE1_PATIENCE)
    phase1_results.append(res)
    print(f"[{i+1}/{len(phase1)}] {rn} -> {res['status']} (rc={res['return_code']})")

print(f'Phase 1 finished in {(time.time()-start)/60:.1f} minutes')

p1_df = leaderboard_df(OUT_DIR)
if len(p1_df):
    p1_df.to_csv(OUT_DIR / 'leaderboard_phase1.csv', index=False)
display(p1_df.head(10))


In [ ]:
# Phase 2: local refinement around top runs
lb = leaderboard_df(OUT_DIR)
if lb.empty:
    raise RuntimeError('No completed runs found. Execute Phase 1 first.')

top = lb.head(3).copy()

def nearby(value, choices):
    if value not in choices:
        return choices[:2]
    idx = choices.index(value)
    left = max(0, idx - 1)
    right = min(len(choices), idx + 2)
    return choices[left:right]

seq_choices = [24, 48, 72, 168]
hid_choices = [32, 64, 128, 256]
layer_choices = [1, 2, 3]
drop_choices = [0.0, 0.05, 0.1, 0.2, 0.3]
lr_choices = [1e-4, 3e-4, 1e-3, 3e-3]
wd_choices = [1e-6, 1e-5, 1e-4]
bs_choices = [16, 32, 64]

candidates = []
for _, r in top.iterrows():
    for seq, hid, lay, dr, lr, wd, bs in itertools.product(
        nearby(int(r['seq_len']), seq_choices),
        nearby(int(r['lstm_hidden']), hid_choices),
        nearby(int(r['lstm_layers']), layer_choices),
        nearby(float(r['dropout']), drop_choices),
        nearby(float(r['learning_rate']), lr_choices),
        nearby(float(r['weight_decay']), wd_choices),
        nearby(int(r['batch_size']), bs_choices),
    ):
        candidates.append((seq, hid, lay, dr, lr, wd, bs))

candidates = list(dict.fromkeys(candidates))
random.shuffle(candidates)
candidates = candidates[:PHASE2_LIMIT]

for i, (seq, hid, lay, dr, lr, wd, bs) in enumerate(candidates):
    cfg = {
        'seq_len': seq,
        'lstm_hidden': hid,
        'lstm_layers': lay,
        'dropout': dr,
        'learning_rate': lr,
        'weight_decay': wd,
        'batch_size': bs,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + 1000 + i,
    }
    rn = run_name_from_cfg('p2', i, cfg)
    res = run_experiment(rn, cfg, max_epochs=PHASE2_MAX_EPOCHS, patience=PHASE2_PATIENCE)
    print(f"[P2 {i+1}/{len(candidates)}] {rn} -> {res['status']} (rc={res['return_code']})")

final_df = leaderboard_df(OUT_DIR)
final_df.to_csv(OUT_DIR / 'leaderboard_final.csv', index=False)
display(final_df.head(15))


In [ ]:
# Final report snippets
final_df = leaderboard_df(OUT_DIR)
if final_df.empty:
    print('No completed runs yet.')
else:
    best = final_df.iloc[0]
    print('Best run:')
    print(best[['run_name', 'val_RMSE', 'test_RMSE', 'val_MAE', 'test_MAE', 'seq_len', 'lstm_hidden', 'lstm_layers', 'dropout', 'learning_rate', 'weight_decay', 'batch_size']])
    print('\nTop 10 by validation RMSE:')
    display(final_df.head(10))

# Optional: zip results for download
# !zip -r runs_st_calnet_colab.zip runs/st_calnet_colab


## Barebones Phase 1 (Smoke Test, Meeting-Safe)

Use this cell for a quick proof that the pipeline works end-to-end.

Characteristics:
- runs a tiny fixed set of 3 lightweight configs
- lower epoch budget and patience
- produces `metrics.json` outputs and a small leaderboard quickly

Recommended when you need results within ~20-60 minutes on a T4.


In [ ]:
# Barebones Phase 1: fixed mini sweep
BARE_P1_MAX_EPOCHS = 8
BARE_P1_PATIENCE = 3

bare_p1_configs = [
    {
        'seq_len': 24,
        'lstm_hidden': 64,
        'lstm_layers': 1,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        'batch_size': 32,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + 500,
    },
    {
        'seq_len': 48,
        'lstm_hidden': 64,
        'lstm_layers': 1,
        'dropout': 0.1,
        'learning_rate': 3e-4,
        'weight_decay': 1e-5,
        'batch_size': 32,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + 501,
    },
    {
        'seq_len': 24,
        'lstm_hidden': 128,
        'lstm_layers': 2,
        'dropout': 0.2,
        'learning_rate': 3e-4,
        'weight_decay': 1e-5,
        'batch_size': 32,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + 502,
    },
]

print('Running barebones Phase 1 with', len(bare_p1_configs), 'configs')
for i, cfg in enumerate(bare_p1_configs):
    rn = run_name_from_cfg('bare_p1', i, cfg)
    res = run_experiment(rn, cfg, max_epochs=BARE_P1_MAX_EPOCHS, patience=BARE_P1_PATIENCE)
    print(f"[{i+1}/{len(bare_p1_configs)}] {rn} -> {res['status']} (rc={res['return_code']})")

bare_p1_lb = leaderboard_df(OUT_DIR)
if not bare_p1_lb.empty:
    bare_p1_lb.to_csv(OUT_DIR / 'leaderboard_barebones_phase1.csv', index=False)

display(bare_p1_lb.head(10))

## Barebones Phase 2 (Quick Refinement)

Use this after barebones Phase 1 to show a refinement cycle quickly.

Characteristics:
- anchors on top 2 current runs
- tests a tiny nearby parameter neighborhood
- capped to at most 6 runs
- produces `leaderboard_barebones_phase2.csv`


In [ ]:
# Barebones Phase 2: tiny local refinement
BARE_P2_MAX_EPOCHS = 12
BARE_P2_PATIENCE = 4
BARE_P2_LIMIT = 6

lb_now = leaderboard_df(OUT_DIR)
if lb_now.empty:
    raise RuntimeError('No completed runs found. Run barebones Phase 1 first.')

top2 = lb_now.head(2).copy()

def near_small(value, choices):
    if value not in choices:
        return choices[:2]
    i = choices.index(value)
    return choices[max(0, i-1):min(len(choices), i+2)]

seq_choices = [24, 48, 72]
hid_choices = [64, 128]
layer_choices = [1, 2]
drop_choices = [0.1, 0.2]
lr_choices = [3e-4, 1e-3]
wd_choices = [1e-5]
bs_choices = [32]

cand = []
for _, r in top2.iterrows():
    for seq, hid, lay, dr, lr, wd, bs in itertools.product(
        near_small(int(r['seq_len']), seq_choices),
        near_small(int(r['lstm_hidden']), hid_choices),
        near_small(int(r['lstm_layers']), layer_choices),
        near_small(float(r['dropout']), drop_choices),
        near_small(float(r['learning_rate']), lr_choices),
        wd_choices,
        bs_choices,
    ):
        cand.append((seq, hid, lay, dr, lr, wd, bs))

cand = list(dict.fromkeys(cand))
random.shuffle(cand)
cand = cand[:BARE_P2_LIMIT]

print('Running barebones Phase 2 with', len(cand), 'configs')
for i, (seq, hid, lay, dr, lr, wd, bs) in enumerate(cand):
    cfg = {
        'seq_len': seq,
        'lstm_hidden': hid,
        'lstm_layers': lay,
        'dropout': dr,
        'learning_rate': lr,
        'weight_decay': wd,
        'batch_size': bs,
        'conv_channels': 64,
        'attn_dim': 64,
        'seed': SEED + 800 + i,
    }
    rn = run_name_from_cfg('bare_p2', i, cfg)
    res = run_experiment(rn, cfg, max_epochs=BARE_P2_MAX_EPOCHS, patience=BARE_P2_PATIENCE)
    print(f"[{i+1}/{len(cand)}] {rn} -> {res['status']} (rc={res['return_code']})")

bare_p2_lb = leaderboard_df(OUT_DIR)
if not bare_p2_lb.empty:
    bare_p2_lb.to_csv(OUT_DIR / 'leaderboard_barebones_phase2.csv', index=False)

display(bare_p2_lb.head(12))